# Day 082 — Exercise 3: Deciding What to Remember

**What you'll build:** `build_extraction_prompt` and `extract_memories` — the model reads a message and returns the durable facts worth storing.

**Why it matters:** you can't save *everything* — most chatter isn't worth remembering. Extraction is the judgement step: pull out the stable facts (name, preferences, goals) and drop the small talk. And like every parser this section, it must degrade to *nothing* rather than crash on messy output.

In [ ]:
import json

def _mock_llm(facts=None, reply='Sure!'):
    """An llm_fn: returns extracted `facts` (as JSON) to the extraction prompt,
    and `reply` to a normal chat prompt - it branches on the system message."""
    payload = json.dumps(facts or {})
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'extract' in system.lower():
            return payload
        return reply
    return _fn
import json

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Task

1. `build_extraction_prompt(message)` — a `system` message instructing the model to return **only** a JSON object of `snake_case_key: value` durable facts (or `{}` if none), and a `user` message with the raw text.
2. `extract_memories(message, llm_fn=None) -> dict` — call the model, `safe_parse_json` the reply (`or {}`), and return `{str(k): str(v) for k, v in data.items()}`. **Never raises.**

## Your Implementation

In [ ]:
def build_extraction_prompt(message):
    """Ask the model which durable facts about the user to store."""
    raise NotImplementedError

def extract_memories(message, llm_fn=None):
    """Return durable facts as {key: value}. Never raises."""
    raise NotImplementedError


In [ ]:

# ── deciding what is worth remembering (LLM-driven) ──────────────────────────
def build_extraction_prompt(message):
    """Ask the model which durable facts about the user to store long-term."""
    system = "\n".join([
        "You extract durable facts about the user that are worth remembering.",
        "Return ONLY a JSON object mapping short snake_case keys to string values.",
        "Store only stable facts - name, location, preferences, goals.",
        "Ignore small talk and one-off questions.",
        "If there is nothing worth remembering, return {}.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": str(message)}]


def extract_memories(message, llm_fn=None):
    """Return durable facts to store as {key: value}. Never raises.

    Unparseable model output falls back to an empty dict, so a bad extraction
    simply stores nothing rather than crashing the turn.
    """
    response = call_llm(build_extraction_prompt(message), llm_fn=llm_fn)
    data = safe_parse_json(response) or {}
    return {str(k): str(v) for k, v in data.items()}


## Automated checks

In [ ]:

score, total = 0, 4
try:
    msgs = build_extraction_prompt("I'm Kutlwano and I love SQL")
    assert msgs[0]['role'] == 'system' and 'extract' in msgs[0]['content'].lower()
    assert msgs[1]['content'] == "I'm Kutlwano and I love SQL"
    score += 1; print("✅ extraction prompt asks for durable facts as JSON")

    facts = extract_memories('x', llm_fn=_mock_llm(facts={'name': 'Kutlwano'}))
    assert facts == {'name': 'Kutlwano'}
    score += 1; print("✅ extract_memories returns the parsed facts")

    empty = extract_memories('hello', llm_fn=_mock_llm(facts={}))
    assert empty == {}
    score += 1; print("✅ nothing worth remembering -> empty dict")

    junk = extract_memories('x', llm_fn=lambda m: 'no json here, sorry')
    assert junk == {}
    score += 1; print("✅ unparseable output -> {} (never raises)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── deciding what is worth remembering (LLM-driven) ──────────────────────────
def build_extraction_prompt(message):
    """Ask the model which durable facts about the user to store long-term."""
    system = "\n".join([
        "You extract durable facts about the user that are worth remembering.",
        "Return ONLY a JSON object mapping short snake_case keys to string values.",
        "Store only stable facts - name, location, preferences, goals.",
        "Ignore small talk and one-off questions.",
        "If there is nothing worth remembering, return {}.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": str(message)}]


def extract_memories(message, llm_fn=None):
    """Return durable facts to store as {key: value}. Never raises.

    Unparseable model output falls back to an empty dict, so a bad extraction
    simply stores nothing rather than crashing the turn.
    """
    response = call_llm(build_extraction_prompt(message), llm_fn=llm_fn)
    data = safe_parse_json(response) or {}
    return {str(k): str(v) for k, v in data.items()}
```

**Why `safe_parse_json(response) or {}`?** The model might wrap its JSON in prose, or emit no JSON at all. `safe_parse_json` returns `None` on failure; `or {}` turns that into an empty result, so a bad extraction stores nothing instead of raising and killing the turn.

</details>